In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from sqlalchemy import create_engine
import os

def connect_to_db():
    engine = create_engine(
        f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@"
        f"{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
    )
    return engine

engine = connect_to_db()

conn = engine.connect()

In [ ]:
df = pd.read_sql("SELECT * FROM public.gold_model_dataset", engine)

df.head()

In [ ]:
num_cols = [
    "stay_duration",
    "advance_booking_days",
    "orig_destination_distance",
    "cnt",
    "total_guests"
]

cat_cols = [
    "is_mobile",
    "is_package",
    "channel",
    "trip_type",
    "srch_destination_type_id"
]

In [ ]:
for col in num_cols:
    plt.figure(figsize=(6,4))
    sns.histplot(df[col], bins=50, kde=True)
    plt.title(f"Distribution of {col}")
    plt.show()

In [ ]:
for col in num_cols:
    plt.figure(figsize=(6,2))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot of {col}")
    plt.show()

In [ ]:
df[num_cols].describe().T

In [ ]:
for col in num_cols:
    p1, p50, p99 = np.percentile(df[col], [1, 50, 99])
    print(f"{col} → P1: {p1:.2f} | Median: {p50:.2f} | P99: {p99:.2f}")

In [ ]:
for col in cat_cols:
    plt.figure(figsize=(8,4))
    df[col].value_counts().plot(kind="bar")
    plt.title(f"Distribution of {col}")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
for col in cat_cols:
    print(f"{col}: {df[col].nunique()}")

In [ ]:
(df["orig_destination_distance"] == -1).sum()

In [ ]:
sns.histplot(df["cnt"], bins=50)
plt.title("Session Intensity (cnt)")
plt.show()

sns.histplot(df["advance_booking_days"], bins=50)
plt.title("Advance Booking Days")
plt.show()

In [ ]:
conn.close()

## Key Insights

- Numerical variables such as `stay_duration`, `advance_booking_days`, and `orig_destination_distance` exhibit skewed distributions, indicating that user behavior is not uniformly distributed.
- Most observations are concentrated in lower ranges for variables like `cnt`, suggesting that the majority of sessions involve limited interaction.
- The presence of long tails in several features indicates that extreme values exist, but likely represent real-world behavior rather than data errors.
- Categorical variables such as `channel`, `trip_type`, and `srch_destination_type_id` show varying levels of cardinality, which may impact downstream analysis.
- A significant proportion of `orig_destination_distance` values are marked as unknown (-1), highlighting the importance of handling this feature carefully.

## Conclusion

The univariate analysis reveals that the dataset contains a mix of skewed numerical variables and moderately distributed categorical features. These patterns are consistent with real user behavior and provide important context for further analysis. The results highlight the need for careful handling of skewness and special values in subsequent steps.